# Reading datamate directories

Results written by datamate (and by flyvis, which builds on it) are directories with a
`_meta.yaml` (`config`, `status`), one HDF5 file per array holding a single dataset `data`,
CSV tables and nested subdirectories. `tensordir.legacy.open_datamate(path)` reads them
without writing to them, so existing results stay usable next to new records. This notebook
writes a small datamate-style tree by hand, reads it, and copies one result into a record.

In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import yaml

import tensordir as td
from tensordir.legacy import open_datamate

tmp = Path(tempfile.mkdtemp(prefix="tensordir-legacy-"))
td.set_root(tmp / "records")

## A datamate tree

The layout a flyvis network directory had: a config, arrays as `.h5` files, a nested
directory of checkpoints, and a CSV table.

In [ ]:
network = tmp / "0000" / "000"
(network / "chkpts").mkdir(parents=True)
(network / "_meta.yaml").write_text(
    yaml.safe_dump({"config": {"type": "NetworkDir", "network": {"n_cells": 5}, "seed": 0}, "status": "done"})
)
for name, value in {"loss": np.linspace(1, 0.1, 20), "cell_types": np.array([b"T4a", b"T4b", b"T5a", b"T5b", b"Mi1"])}.items():
    with h5py.File(network / f"{name}.h5", "w") as f:
        f.create_dataset("data", data=value)
with h5py.File(network / "chkpts" / "chkpt_00010.h5", "w") as f:
    f.create_dataset("data", data=np.ones((5, 5)))
pd.DataFrame({"chkpt": [10], "val_loss": [0.2]}).to_csv(network / "validation.csv", index=False)

sorted(str(p.relative_to(tmp)) for p in tmp.rglob("*"))

## Open it

A `DatamateDirectory` is a read-only mapping. Keys drop the `.h5` and `.csv` suffixes; `.h5`
arrays load as numpy (byte strings decoded), CSV files as DataFrames, subdirectories as
`DatamateDirectory` objects, and other files as paths. HDF5 files are opened with file locking
disabled, which avoids hangs on NFS and Lustre. Attribute access works as in datamate.

In [ ]:
old = open_datamate(network)
print(old, old.status, old.type)
print(old.config)
print(list(old), old.cell_types, old.loss[:3])
print(old.chkpts, list(old.chkpts), old.chkpts["chkpt_00010"].shape)
old.validation

## Copy a result into a record

There are no import aliases for datamate's API: a record type is a different object model.
To migrate a result, make its config the settings of a record type and copy the arrays in
`__init__`. The datamate directory enters as an input path, so its location is part of the key.

In [ ]:
class Network(td.Record):
    def __init__(self, source: str, /, *, n_cells: int, seed: int = 0):
        legacy = open_datamate(source)
        self.loss = legacy.loss
        self.cell_types = legacy.cell_types.tolist()  # JSON
        self["chkpt_00010"] = legacy.chkpts["chkpt_00010"]
        self.validation = legacy.validation


config = old.config
migrated = Network(str(network), n_cells=config["network"]["n_cells"], seed=config["seed"])
migrated.tree()
print(migrated.cell_types, migrated.load("loss")[-1])

See [Migrating](../../migrating/) for how datamate and flyvis idioms map to
tensordir.